# Generate within eddy time series - take mean and standard deviation of PACE variables within eddy

Author: SEL, eddy functions by LJK

In [1]:
import math, pylab, csv
import xarray as xr
import numpy as np
from datetime import datetime
from datetime import date
from itertools import groupby
from collections import Counter
from matplotlib.path import Path
import matplotlib.pyplot as plt
import earthaccess
import xarray as xr
from xarray.backends.api import open_datatree
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np
#%matplotlib widget
from scipy.ndimage import generic_filter
from scipy.ndimage import gaussian_filter
from scipy.interpolate import griddata
import seaborn as sns 
import pandas
from xarray.backends.api import open_datatree
import numpy as np
import os

In [2]:
# %matplotlib widget

In [3]:
fontsize = 20

plt.rc('font', size=fontsize)          # controls default text sizes
plt.rc('axes', titlesize=fontsize)     # fontsize of the axes title
plt.rc('axes', labelsize=fontsize)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=fontsize)    # fontsize of the tick labels
plt.rc('ytick', labelsize=fontsize)    # fontsize of the tick labels
plt.rc('legend', fontsize=fontsize)    # legend fontsize
plt.rc('figure', titlesize=fontsize)  # fontsize of the figure title

In [ ]:
moana_location='/home/jovyan/GO-SWACE/data/moana' #here, add your directory where all MOANA netCDF images are stored. Must precrop before running SeaDAS on Level-1 

In [4]:
ds = xr.open_dataset('Edward_Eddy_trajectory_nrt_3.2exp_cyclonic_20180101_20240723.nc')
ds

<xarray.Dataset> Size: 202kB
Dimensions:                        (obs: 223, NbSample: 20)
Dimensions without coordinates: obs, NbSample
Data variables: (12/27)
    amplitude                      (obs) float64 2kB ...
    effective_area                 (obs) float32 892B ...
    effective_contour_height       (obs) float32 892B ...
    effective_contour_latitude     (obs, NbSample) float64 36kB ...
    effective_contour_longitude    (obs, NbSample) float64 36kB ...
    effective_contour_shape_error  (obs) float64 2kB ...
    ...                             ...
    speed_contour_longitude        (obs, NbSample) float64 36kB ...
    speed_contour_shape_error      (obs) float64 2kB ...
    speed_radius                   (obs) float64 2kB ...
    time                           (obs) datetime64[ns] 2kB ...
    track                          (obs) uint32 892B ...
    uavg_profile                   (obs, NbSample) float64 36kB ...
Attributes: (12/19)
    Metadata_Conventions:      Unidata Dataset Discovery v1.0
    comment:                   Surface product; mesoscale eddies
    creator_email:             aviso@altimetry.fr
    creator_url:               https://www.aviso.altimetry.fr
    date_created:              2024-08-06T09:38:06Z
    framework_used:            https://github.com/AntSimi/py-eddy-tracker
    ...                        ...
    standard_name_vocabulary:  NetCDF Climate and Forecast (CF) Metadata Conv...
    summary:                   This dataset contains eddy atlas from all-sate...
    time_coverage_duration:    P2396D
    time_coverage_end:         2024-07-23T00:00:00Z
    time_coverage_start:       2018-01-01T00:00:00Z
    title:                     Mesoscale Cyclonic Eddies in Altimeter Observa...

In [5]:
def in_eddy(ds,float_lat,float_lon,float_time):
    """
    float_lat: degrees north
    float_lon: degrees east
    float_time: should be in format 'YYYY-MM-DD'
    """

    def all_equal(iterable):
        g = groupby(iterable)
        return next(g, True) and not next(g, False)

    in_eddy_flag = False 

    float_time = np.datetime64(float_time)
    if float_time in ds.time: # some dates not in dateset        
        for i in np.where(ds.time == float_time)[0]: # usually will only be 1 eddy, but sometimes there are 2 after a split
            contour_lons = np.array(ds.effective_contour_longitude[i]) # eddy lons
            contour_lats = np.array(ds.effective_contour_latitude[i]) # eddy lats
            
            if all_equal(contour_lons): # eddy break
                pass
            else:
                poly = Path([(contour_lats[j],contour_lons[j]) for j in np.arange(0,len(contour_lats))]) # set up the polygon
                if poly.contains_points([(float_lat,float_lon)]): #find if point is inside the polygon
                    in_eddy_flag = True 

    return in_eddy_flag

In [7]:
satdate,N,pico_mean,pico_std,pro_mean,pico_mean,syn_mean,syn_std=[],[],[],[],[],[],[],[];

In [8]:
files=[]
for filename in os.listdir(moana_location):
    if filename.endswith('.nc'):
        files.append(filename)

In [9]:
files

['PACE_OCI.20240429T172138.L2_MOANA.V2.nc',
 'PACE_OCI.20240502T172807.L2_MOANA.V2.nc',
 'PACE_OCI.20240504T165938.L2_MOANA.V2.nc',
 'PACE_OCI.20240426T171515.L2_MOANA.V2.nc']

In [10]:
dates = [i for i in aviso_ds.time.values] # these are sorted already
print(min(dates))
print(max(dates)) # still existed at the end of the dataset
tminny=str(min(dates))
tminny=tminny[0:10]
tmaxxy=str(max(dates))
tmaxxy=tmaxxy[0:10]
tmin,tmax = tminny,tmaxxy
center_lats = [i for i in aviso_ds.latitude.values]
center_lons = [i for i in aviso_ds.longitude.values]
print(len(center_lats))
print(min(center_lats))
print(max(center_lats))
print(min(center_lons)-360)
print(max(center_lons)-360)
#Define box based on eddy
latmin,latmax = min(center_lats)-2,max(center_lats)+2
lonmin,lonmax = min(center_lons)-360-2,max(center_lons)-360+2

bbox = (lonmin+2, latmin+2, lonmax-2, latmax-2)

In [11]:
picomean=[]
picostd=[]
promean=[]
prostd=[]
synmean=[]
synstd=[]
chlmean=[]
chlstd=[]
pocmean=[]
pocstd=[]

In [12]:
for IDX in range(0,len(files)): #FOR EACH IMAGE. had to do in chunks 
    datatree = open_datatree('data/' + files[IDX])
    dataset = xr.merge(datatree.to_dict().values())
    dataset = dataset.set_coords(("longitude", "latitude"))
    h=str(files[IDX])
    sat_date=h[9:17]
    mask = (dataset.longitude >= bbox[0]) & (dataset.longitude <= bbox[2]) & (dataset.latitude >= bbox[1]) & (dataset.latitude <= bbox[3])
    if mask.sum() == 0:
        mask = (dataset.longitude >= bbox[0]) & (dataset.longitude <= bbox[2])
        if mask.sum() == 0:
            print('Mask via longitude only')
            mask = (dataset.latitude >= bbox[1]) & (dataset.latitude <= bbox[3])
            print(mask.sum())
        else:
            print('Mask via latitude only')
            print(mask.sum())
    ds_masked = dataset.where(mask, drop=True)
    lat=ds_masked.latitude.values.flatten()
    lon=ds_masked.longitude.values.flatten()+360
    pico=ds_masked.picoeuk_moana.values.flatten()
    pro=ds_masked.prococcus_moana.values.flatten()
    syn=ds_masked.syncoccus_moana.values.flatten()
    chl=ds_masked.chlor_a.values.flatten()
    poc=ds_masked.poc.values.flatten()

    year=sat_date[0:4]
    month=sat_date[4:6]
    day=sat_date[6:8]

    time=year+'-'+month+'-'+day
        
    pico_inside,pro_inside,syn_inside,chl_inside,poc_inside=[],[],[],[],[]
    
    for index in np.arange(0,len(lat)):
        val=in_eddy(ds,lat[index],lon[index],time)
        if val:
            pico_inside.append(pico[index]) 
            pro_inside.append(pro[index])
            syn_inside.append(syn[index])
            chl_inside.append(chl[index])
            poc_inside.append(poc[index])
    
    N1=np.size(np.where(np.isnan(pico_inside)==False))

    if N1>3000:
        pico_mean=np.nanmean(pico_inside);
        pico_std=np.nanstd(pico_inside);
        pro_mean=np.nanmean(pro_inside);
        pro_std=np.nanstd(pro_inside);
        syn_mean=np.nanmean(syn_inside);
        syn_std=np.nanstd(syn_inside);
        chl_mean=np.nanmean(chl_inside);
        chl_std=np.nanstd(chl_inside);
        poc_mean=np.nanmean(poc_inside);
        poc_std=np.nanstd(poc_inside);

        chlmean.append(chl_mean);
        pocmean.append(poc_mean);
        picomean.append(pico_mean);
        promean.append(pro_mean);
        synmean.append(syn_mean);
        
        chlstd.append(chl_std);
        pocstd.append(poc_std);
        picostd.append(pico_std);
        prostd.append(pro_std);
        synstd.append(syn_std);
        N.append(N1);
        satdate.append(sat_date);

In [13]:
N1

10042

In [14]:
d = {'satdate': satdate, 'N':N, 'chlmean': chlmean, 'chlstd': chlstd, 'pocmean': pocmean, 'pocstd': pocstd,
    'picomean':picomean, 'picostd':picostd, 'promean':promean, 'prostd':prostd, 'synmean':synmean, 'synstd':synstd}
data = pandas.DataFrame(data=d)
data

,satdate,N,chlmean,chlstd,pocmean,pocstd,picomean,picostd,promean,prostd,synmean,synstd
0,20240429,13694,0.270173,0.108355,78.408867,18.989225,5874.011230,2478.893799,410659.25000,57472.832031,24705.712891,10214.199219
1,20240502,14700,0.269078,0.080546,75.276474,14.916386,8446.411133,4327.921875,238928.84375,75568.703125,13307.099609,6844.752930
2,20240504,8025,0.231511,0.061017,67.615486,10.743677,6123.258301,2157.413330,262917.03125,78419.054688,17115.373047,11896.685547
3,20240426,10042,0.223379,0.054780,67.453712,11.131034,16727.558594,8238.766602,397549.43750,65839.765625,59852.359375,33853.710938


In [21]:
data.to_csv('PACE_TIMESERIES_MOANA_CHL_POC.csv')

In [16]:
IDX

3

In [20]:
data_sorted = data.sort_values(by='satdate')

In [18]:
data_sorted

,satdate,N,chlmean,chlstd,pocmean,pocstd,picomean,picostd,promean,prostd,synmean,synstd
3,20240426,10042,0.223379,0.054780,67.453712,11.131034,16727.558594,8238.766602,397549.43750,65839.765625,59852.359375,33853.710938
0,20240429,13694,0.270173,0.108355,78.408867,18.989225,5874.011230,2478.893799,410659.25000,57472.832031,24705.712891,10214.199219
1,20240502,14700,0.269078,0.080546,75.276474,14.916386,8446.411133,4327.921875,238928.84375,75568.703125,13307.099609,6844.752930
2,20240504,8025,0.231511,0.061017,67.615486,10.743677,6123.258301,2157.413330,262917.03125,78419.054688,17115.373047,11896.685547
